# OptiVision RAG — ViDoRe / ColPali ablation on a free GPU

Produces the same ablation table as `reports/colsmol/benchmark.md`, but with the
real ColPali-v1.3 encoder over real ViDoRe benchmark pages instead of ColSmol-256M
over generated pages. This is the single result that upgrades the paper from
"measured on a laptop" to "measured on the benchmark everyone else reports".

**Runtime:** Colab `T4 GPU` (free tier) or Kaggle `GPU T4 x2`. Set it *before* running:
Colab → Runtime → Change runtime type → T4 GPU.

**Expected cost:** ColPali-v1.3 is ~3B parameters in bfloat16 (~6 GB VRAM), which fits
a T4's 15 GB. Encoding runs at roughly 1–3 pages/second, so 500 pages takes 5–10 minutes
and the whole notebook finishes inside one free session. The ablation rows after encoding
are near-instant because they replay over the cached vectors.

**Before you start:** push the local repo so this notebook can clone the current code.
The `docs/IMPROVEMENTS.md` work (blocked scoring, int8 rescaling) must be committed —
without it the memory figures in the paper do not match the code.

## 1. Check the GPU

If this prints `no GPU`, stop and change the runtime type — everything below will
otherwise fall back to CPU and take hours.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU — change the runtime type"

## 2. Install

`colpali-engine` pins a `transformers` range; installing it last lets pip resolve to a
version that actually loads the checkpoint. Colab ships a compatible torch already, so
we do not reinstall it.

**The torchao line is not an optimisation — the run dies without it.** ColPali ships
as a PEFT LoRA adapter over PaliGemma, so `from_pretrained` calls `load_adapter` and
PEFT walks its LoRA dispatcher chain. `dispatch_torchao` calls `is_torchao_available()`
*before* it can decide it does not apply, and recent PEFT hard-raises on
torchao < 0.16 — which is exactly what the Colab image ships (0.10.0). The run then
fails with `ImportError: Found an incompatible version of torchao` on a library
nothing here uses. Removing the package makes that check return `False` via
`find_spec` instead of raising, and the dispatcher falls through to the normal
`Linear` path.

Do not "fix" it by upgrading torchao instead: `torchao>=0.16` pulls a matching torch
build, which is a multi-GB reinstall and a CUDA mismatch risk in the middle of a
timed session, to satisfy a dependency this notebook never calls.

(Since `configs/colpali.yaml` now points at the *merged* ColPali weights, no adapter
is injected and this dispatcher never runs. The uninstall stays because it costs
nothing and anyone who repoints the config at an adapter checkpoint will hit the gate
immediately.)


In [ ]:
import os
import shutil

REPO = "https://github.com/Daemon-VI/optivision-rag.git"

# Colab puts you in /content, Kaggle in /kaggle/working. Kaggle matters here:
# Colab's free GPU quota runs out well before four splits are done, and Kaggle
# gives 30 GPU-hours a week. Enable Internet in the notebook settings there, or
# the clone and the Hub downloads both fail.
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
WORKDIR = f"{BASE}/optivision"

# Re-running this cell must leave you on the current commit. A bare `git clone`
# into an existing directory fails, and with `%cd` already inside it the notebook
# then quietly carries on against whatever code was there before - which is how
# you end up reading a traceback whose line numbers no longer match the source.
#
# Update in place rather than re-cloning: `reset --hard` does not touch untracked
# files, so data/ and the encode caches survive a code update. Losing those means
# re-encoding the corpus, which is the expensive part of this notebook.
%cd {BASE}
if os.path.isdir(f"{WORKDIR}/.git"):
    !git -C {WORKDIR} fetch --depth 1 origin main
    !git -C {WORKDIR} reset --hard FETCH_HEAD
else:
    if os.path.isdir(WORKDIR):
        shutil.rmtree(WORKDIR)
    !git clone --depth 1 {REPO} {WORKDIR}
%cd {WORKDIR}

!pip install -q -e ".[bench]"
!pip install -q "colpali-engine>=0.3.10" "transformers>=4.46"

# See the note above: PEFT's LoRA dispatcher version-gates on torchao, and the
# Colab image ships a version below its floor. Nothing here is torchao-quantized.
!pip uninstall -y -q torchao

print("installed at", end=" ")
!git -C {WORKDIR} rev-parse --short HEAD

In [ ]:
import os

# Only lifts Hub rate limits, but the merged ColPali weights are a ~6 GB download
# on every fresh runtime, and an unauthenticated fetch is throttled hard.
# Colab: the key icon in the sidebar. Kaggle: Add-ons -> Secrets.
token = None
try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

if token:
    os.environ["HF_TOKEN"] = token
    print("HF_TOKEN set")
else:
    print("no HF_TOKEN - continuing unauthenticated (slower, rate-limited)")

# Bound the Hub metadata calls so a flaky network cannot wedge the run.
os.environ["HF_HUB_ETAG_TIMEOUT"] = "10"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
import importlib.util
from importlib.metadata import version

import torch

from optivision.config import Config

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# torchao must be gone before the encoder loads. Fail here, where the cause is
# obvious, rather than ten minutes into the encode pass with a PEFT traceback.
if importlib.util.find_spec("torchao") is not None:
    raise SystemExit("torchao is still installed - re-run the install cell")

# Prove the checkout is current before spending an hour on it. The adapter-only
# checkpoint silently loads a randomly initialised projection head, so running
# stale code here does not merely waste the session - it produces numbers.
checkpoint = Config.load("configs/colpali.yaml").encoder.model_name
if not checkpoint.endswith("-merged"):
    raise SystemExit(
        f"stale checkout: configs/colpali.yaml still names {checkpoint}. "
        "Re-run the install cell - it should have re-cloned."
    )

# Record what actually resolved. "Whatever Colab shipped that month" is not a
# reproducibility section: paste these into the paper next to the commit hash.
print()
for pkg in ("torch", "transformers", "peft", "colpali-engine", "datasets"):
    print(f"{pkg:<16} {version(pkg)}")
print(f"{'torchao':<16} (removed - PEFT LoRA dispatcher version gate)")
print(f"{'checkpoint':<16} {checkpoint}")
print()
!git rev-parse --short HEAD

## 3. Fetch a ViDoRe split

`fetch-vidore` materialises a split as page images plus a `queries.json` with exact
ground truth, in the same shape the generated corpus uses — so the benchmark runner
cannot tell them apart, and the ablation is identical apart from the data.

Start with **one** split to confirm the pipeline runs end to end, then come back and
loop over the rest in section 7. Useful splits:

| dataset | character |
|---|---|
| `vidore/docvqa_test_subsampled` | scanned industry documents — the hard, realistic one |
| `vidore/syntheticDocQA_energy_test` | clean synthetic reports — closest to our generated corpus |
| `vidore/infovqa_test_subsampled` | infographics — dense colour, little whitespace |
| `vidore/tabfquad_test_subsampled` | tables — the case redundancy pruning targets |

`infovqa` is the interesting adversarial case for this paper: our whole premise is that
a page is mostly blank paper, and an infographic is not. If spatial pruning holds up
there too, the claim is much stronger; if it collapses, that belongs in the limitations.

In [ ]:
DATASET = "vidore/docvqa_test_subsampled"
LIMIT = 500  # pages; raise once you know the timing on your session

!optivision fetch-vidore --dataset {DATASET} --out data/vidore --limit {LIMIT}

## 4. Point the config at ColPali on the GPU

`configs/colpali.yaml` in the repo already names the checkpoint and bfloat16, but it
indexes into Qdrant. For measurement we override the index to the exact brute-force
backend — the same control the paper describes, so that an ANN structure cannot
contribute its own recall loss to the numbers.

**On the checkpoint.** The config names `vidore/colpali-v1.3-merged`, not
`vidore/colpali-v1.3`. The latter is published adapter-only — `adapter_model.safetensors`
with no `config.json` — so loading it makes transformers inject a LoRA adapter over the
PaliGemma base. On current transformers/peft builds the adapter's key prefixes do not
match, and the projection head is left **randomly initialised** instead of loaded:

    base_model.model.custom_text_proj.lora_A.default.weight | UNEXPECTED
    custom_text_proj.lora_A.default.weight                  | MISSING

The merged repo is the same model with the LoRA already folded into the weights, so
there is no injection step and nothing to mis-key. The encoder now also refuses to run
if any parameter is still on the meta device, which is what an unloaded weight looks
like — a benchmark table produced by a randomly initialised head would look completely
normal and mean nothing.


In [ ]:
from pathlib import Path

import yaml

cfg = yaml.safe_load(Path("configs/colpali.yaml").read_text())
cfg["index"] = {"backend": "numpy", "path": "data/index/colpali", "on_disk": True}
cfg["encoder"]["batch_size"] = 4        # T4-safe; raise to 8 on an A100/L4
cfg["encoder"]["max_pages_in_flight"] = 8

Path("configs/colpali_bench.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
print(yaml.safe_dump(cfg, sort_keys=False))

## 5. Run the ablation

The corpus is encoded **once** into `--cache`; every ablation row then replays over
those identical vectors. That is what makes this a controlled experiment rather than
a leaderboard, and it also means a crashed session is cheap to resume — rerun this
cell and the encode pass is skipped.

Keep the cache on Drive if you are on Colab and worried about disconnects (section 8).

In [ ]:
!optivision bench \
    data/vidore/images data/vidore/queries.json \
    -c configs/colpali_bench.yaml \
    --out reports/colpali \
    --sweep \
    --cache data/cache/colpali.npz

## 6. The table, in the form the paper wants

Prints the markdown table and the LaTeX rows. Paste the LaTeX straight into
`paper/optivision.tex` — the column order matches Table I exactly.

In [ ]:
import json
from pathlib import Path

report = json.loads(Path("reports/colpali/benchmark.json").read_text())
print(Path("reports/colpali/benchmark.md").read_text())

print("\n% ---- LaTeX rows for paper/optivision.tex, Table I ----")
for r in report["rows"]:
    print(
        f"{r['variant']} & {r['note']} & {r['tokens_per_page']:.1f} & "
        f"{r['kb_per_page']:.2f} & ${r['compression_ratio']:.1f}\\times$ & "
        f"{r['ndcg@5']:.4f} & {r['recall@1']:.4f} & {r['hit@5']:.4f} & "
        f"{100 * r['ndcg5_retention']:.1f}\\% & "
        f"{r['kendall_tau_vs_baseline']:.3f} \\\\"
    )

### The three claims, re-checked on real data

This cell restates the paper's findings as arithmetic on the new numbers, so you can
see immediately whether they survived the change of encoder and corpus. **If any of
them flips, that is a result, not a failure** — the paper's contribution is the
attribution, and "the asymmetry disappears at 3B scale" would be just as publishable
as the asymmetry itself, provided you report what you actually measured.

In [ ]:
rows = {r["variant"]: r for r in report["rows"]}


def pct_loss(variant: str) -> float:
    return 100 * (1 - rows[variant]["ndcg5_retention"])


prune_cost = pct_loss("spatial+redundancy")
binary_cost = pct_loss("binary-only")
int8_cost = pct_loss("int8-only")

print(f"pruning only    : -{prune_cost:5.1f}% nDCG@5   "
      f"(tau {rows['spatial+redundancy']['kendall_tau_vs_baseline']:.3f}, "
      f"{rows['spatial+redundancy']['compression_ratio']:.1f}x)")
print(f"binary only     : -{binary_cost:5.1f}% nDCG@5   "
      f"(tau {rows['binary-only']['kendall_tau_vs_baseline']:.3f}, "
      f"{rows['binary-only']['compression_ratio']:.1f}x)  <- no tokens dropped")
print(f"int8 only       : -{int8_cost:5.1f}% nDCG@5   "
      f"(tau {rows['int8-only']['kendall_tau_vs_baseline']:.3f}, "
      f"{rows['int8-only']['compression_ratio']:.1f}x)")

print()
print(f"Claim 1  pruning is close to free           : "
      f"{'HOLDS' if prune_cost < 6 else 'DOES NOT HOLD'} ({prune_cost:.1f}% loss)")
print(f"Claim 3  the loss lives in the quantizer    : "
      f"{'HOLDS' if binary_cost > 2 * prune_cost else 'DOES NOT HOLD'} "
      f"(binary {binary_cost:.1f}% vs pruning {prune_cost:.1f}%)")

sweep = [v for v in rows if v.startswith("keep-")]
if sweep:
    span = max(100 * rows[v]["ndcg5_retention"] for v in sweep) - \
           min(100 * rows[v]["ndcg5_retention"] for v in sweep)
    print(f"Claim 2  token budget barely matters       : "
          f"{'HOLDS' if span < 2 else 'DOES NOT HOLD'} "
          f"(sweep spans {span:.1f} points of retention)")

### Regenerate the paper figures on the new numbers

In [ ]:
import shutil
from pathlib import Path

# make_figs.py reads reports/colsmol/benchmark.json; point it at the new run
Path("reports/colsmol").mkdir(parents=True, exist_ok=True)
shutil.copy("reports/colpali/benchmark.json", "reports/colsmol/benchmark.json")

!pip install -q matplotlib
!python paper/make_figs.py

from IPython.display import Image, display

!python -c "import pathlib,re; p=pathlib.Path('paper/make_figs.py'); s=p.read_text(); pathlib.Path('paper/_png.py').write_text(s.replace('.pdf\"', '.png\"'))"
!python paper/_png.py
display(Image("paper/figs/tradeoff.png"), Image("paper/figs/sweep.png"))

## 7. All four splits

Run this only after section 5 has succeeded once. Reporting several ViDoRe splits is
what makes the result a benchmark number rather than an anecdote, and the per-split
spread is itself informative: whitespace-heavy splits should prune better than
infographics, and if they do not, the pixel saliency detector is not doing what the
paper says it does.

Each split gets its own encode cache, so a crash costs you one split, not all of them.

In [ ]:
SPLITS = [
    "vidore/docvqa_test_subsampled",
    "vidore/syntheticDocQA_energy_test",
    "vidore/infovqa_test_subsampled",
    "vidore/tabfquad_test_subsampled",
]
PER_SPLIT_LIMIT = 500

# Build each command as a single Python string and run it with !{cmd}.
# `$tag` inside a multi-line `!` continuation is not interpolated - it expands to
# nothing, and the run dies on a path like data/vidore_/queries.json after the
# fetch has already succeeded. An f-string leaves nothing for the shell to guess.
RULE = "=" * 70

for split in SPLITS:
    tag = split.split("/")[-1]
    print()
    print(RULE)
    print(split)
    print(RULE)

    fetch = (
        f"optivision fetch-vidore --dataset {split} "
        f"--out data/vidore_{tag} --limit {PER_SPLIT_LIMIT}"
    )
    bench = (
        f"optivision bench data/vidore_{tag}/images data/vidore_{tag}/queries.json "
        f"-c configs/colpali_bench.yaml --out reports/colpali_{tag} --sweep "
        f"--cache data/cache/colpali_{tag}.npz"
    )
    print(f"$ {fetch}")
    !{fetch}
    print(f"$ {bench}")
    !{bench}

In [ ]:
# Cross-split summary — the table for a "Results across ViDoRe splits" subsection.
import json
from pathlib import Path

print(f"{'split':<34} {'prune loss':>11} {'binary loss':>12} {'int8 loss':>10} {'full compr.':>12}")
print("-" * 82)
for split in SPLITS:
    tag = split.split("/")[-1]
    path = Path(f"reports/colpali_{tag}/benchmark.json")
    if not path.exists():
        print(f"{tag:<34} {'(not run)':>11}")
        continue
    rows = {r["variant"]: r for r in json.loads(path.read_text())["rows"]}
    loss = lambda v: 100 * (1 - rows[v]["ndcg5_retention"])  # noqa: E731
    print(f"{tag:<34} {loss('spatial+redundancy'):>10.1f}% "
          f"{loss('binary-only'):>11.1f}% {loss('int8-only'):>9.1f}% "
          f"{rows['optivision']['compression_ratio']:>11.1f}x")

## 8. Save the results off the ephemeral disk

Colab discards everything when the session ends. Download the reports before you close
the tab — they are small (a few hundred KB) and they are the entire point of the run.

**Commit `reports/colpali*/` to the repo.** The paper claims the benchmark is
reproducible, and a reviewer who clones the repo should find the numbers that are
printed in it.

In [ ]:
import shutil

archive = shutil.make_archive(f"{BASE}/optivision_colpali_reports", "zip", "reports")
print("wrote", archive)

try:
    from google.colab import files

    files.download(archive)
except ImportError:
    # On Kaggle, anything under /kaggle/working is saved with the notebook version
    # and downloadable from the Output tab - nothing more to do here.
    print("not on Colab - collect the archive from the output/working directory")

---

## What to change in the paper afterwards

1. **Table I** — replace the rows with the LaTeX printed in section 6.
2. **Section IV-A** — encoder becomes `vidore/colpali-v1.3`, bfloat16, on a T4;
   corpus becomes the ViDoRe split(s) and their page/query counts.
3. **Section IV-B** — the precise/topical query split is a property of *our* generated
   corpus. ViDoRe queries do not divide that way, so either drop the subsection or
   restate it as a property of the earlier experiment.
4. **Limitations** — delete "corpus is generated, not scanned" and "ColSmol is not
   ColPali". These are the two limitations the run exists to remove, and leaving them
   in after removing them reads as carelessness.
5. **Abstract and Conclusion** — update every number quoted in prose. There are about a
   dozen; grep the `.tex` for `\%` and `$\tau` and check each one.
6. **Keep the ColSmol table** as a secondary result if the findings agree — a claim that
   holds at both 256M and 3B is stronger than one measured once. Report it as a
   scale-robustness check, not as the headline.

If a finding *changes*, say so plainly and report both. A paper that says "we expected
X, measured Y, here is why" is more useful than one that only ever confirms itself, and
reviewers are considerably better at spotting a buried disagreement than at spotting an
honest one.